In [ ]:
%%writefile vector_add.cu

#include <iostream>
#include <cuda_runtime.h>
#include <limits>

using namespace std;

// CUDA Kernel
__global__ void add(int* A, int* B, int* C, int size) {
    int tid = blockIdx.x * blockDim.x + threadIdx.x;

    if (tid < size) {
        C[tid] = A[tid] + B[tid];
    }
}

// Print vector
void print(int* vec, int size) {
    for (int i = 0; i < size; i++) {
        cout << vec[i] << " ";
    }
    cout << endl;
}

// Safe integer input
int getInt(string message) {
    int value;

    while (true) {
        cout << message;
        cin >> value;

        if (cin.fail()) {
            cout << "Invalid input! Enter a number.\n";

            cin.clear();
            cin.ignore(numeric_limits<streamsize>::max(), '\n');
        }
        else if (value <= 0) {
            cout << "Value must be > 0.\n";
        }
        else {
            return value;
        }
    }
}

// Safe vector input
void inputVector(int* vec, int size, string name) {

    cout << "Enter elements of " << name << ":\n";

    for (int i = 0; i < size; i++) {

        while (true) {

            cout << name << "[" << i << "] = ";
            cin >> vec[i];

            if (cin.fail()) {
                cout << "Invalid input! Enter a number.\n";

                cin.clear();
                cin.ignore(numeric_limits<streamsize>::max(), '\n');
            }
            else {
                break;
            }
        }
    }
}

int main() {

    int N;

    // Input size
    N = getInt("Enter size of vectors: ");

    size_t bytes = N * sizeof(int);

    int *A = new int[N];
    int *B = new int[N];
    int *C = new int[N];

    // Input vectors
    inputVector(A, N, "A");
    inputVector(B, N, "B");

    cout << "\nVector A: ";
    print(A, N);

    cout << "Vector B: ";
    print(B, N);

    // Device memory
    int *d_A, *d_B, *d_C;

    cudaMalloc(&d_A, bytes);
    cudaMalloc(&d_B, bytes);
    cudaMalloc(&d_C, bytes);

    cudaMemcpy(d_A, A, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, B, bytes, cudaMemcpyHostToDevice);

    int threadsPerBlock = 256;
    int blocksPerGrid = (N + threadsPerBlock - 1) / threadsPerBlock;

    // Kernel launch
    add<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, N);

    cudaDeviceSynchronize();

    cudaMemcpy(C, d_C, bytes, cudaMemcpyDeviceToHost);

    cout << "Addition Result: ";
    print(C, N);

    // Cleanup
    delete[] A;
    delete[] B;
    delete[] C;

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    return 0;
}

Writing vector_add.cu


In [ ]:
!nvidia-smi

Wed May 13 10:54:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
!nvcc vector_add.cu -o vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./vector_add

Enter size of vectors: 5
Enter elements of A:
A[0] = 1
A[1] = 2
A[2] = 3
A[3] = 4
A[4] = 5
Enter elements of B:
B[0] = 10
B[1] = 20
B[2] = 30
B[3] = 40
B[4] = 50

Vector A: 1 2 3 4 5 
Vector B: 10 20 30 40 50 
Addition Result: 11 22 33 44 55 
